In [6]:
import os
import zipfile
import pandas as pd
from google.colab import drive
from sklearn.model_selection import train_test_split
from collections import Counter
from PIL import Image

In [7]:
# 1. 드라이브 마운트 & 압축 해제
drive.mount('/content/drive/')

zip_path = '/content/drive/MyDrive/오즈코딩/13_해커톤/open.zip'
data_dir = '/content/chest_xray'

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(data_dir)

Mounted at /content/drive/


In [8]:
# 2. CSV 로드
train_df = pd.read_csv(f'{data_dir}/train.csv')
test_df = pd.read_csv(f'{data_dir}/test.csv')
sample_submission = pd.read_csv(f'{data_dir}/sample_submission.csv')

In [9]:
# 3. CSV ↔ 실제 이미지 파일명 일치 확인
for name, df in [('train', train_df), ('test', test_df)]:
    files = set(os.listdir(f'{data_dir}/{name}'))
    print(f'{name}: CSV {len(df)}행 / 이미지 {len(files)}장 / 파일명 일치 {set(df["file_name"]) == files}')

train: CSV 5216행 / 이미지 5216장 / 파일명 일치 True
test: CSV 624행 / 이미지 624장 / 파일명 일치 True


In [10]:
# 4. 클래스 분포 (개수 + 비율)
dist = train_df['label'].value_counts().sort_index()
print(pd.DataFrame({'count': dist, 'ratio(%)': (dist / dist.sum() * 100).round(2)}))

       count  ratio(%)
label                 
0       1341     25.71
1       3875     74.29


In [11]:
# 5. Train / Validation 분할 (label 비율 유지)
train_split, val_split = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df['label']
)
print(f'Train: {len(train_split)} / Validation: {len(val_split)} / Test: {len(test_df)}')

Train: 4172 / Validation: 1044 / Test: 624


In [12]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os

# ResNet18을 학습하기 위한 기본 이미지 전처리
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 데이터셋 클래스
class ChestXrayDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(
            self.image_dir, row['file_name']
        )

        image = Image.open(image_path).convert('L')

        if self.transform:
            image = self.transform(image)

        if 'label' in self.df.columns:
            label = int(row['label'])
            return image, label

        return image

# 데이터셋 생성
train_dataset = ChestXrayDataset(
    train_split, '/content/chest_xray/train', transform
)

val_dataset = ChestXrayDataset(
    val_split, '/content/chest_xray/train', transform
)

# DataLoader 생성
train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True
)

val_loader = DataLoader(
    val_dataset, batch_size=32, shuffle=False
)

# 입력 형태 확인
images, labels = next(iter(train_loader))

print("이미지 형태:", images.shape)
print("레이블 형태:", labels.shape)

이미지 형태: torch.Size([32, 3, 224, 224])
레이블 형태: torch.Size([32])


In [13]:
import torch.nn as nn
from torchvision import models

# GPU 사용 여부 확인
device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

# 사전 학습된 ResNet18 불러오기
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

# 마지막 분류층을 정상 / 폐렴 2개 클래스로 변경
model.fc = nn.Linear(model.fc.in_features, 2)

# GPU 또는 CPU로 모델 이동
model = model.to(device)

print("사용 장치:", device)
print(model.fc)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 179MB/s]


사용 장치: cuda
Linear(in_features=512, out_features=2, bias=True)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. 손실 함수와 옵티마이저
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 2. 학습 설정
num_epochs = 5

for epoch in range(num_epochs):

    # 학습 단계
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)

        train_correct += (predicted == labels).sum().item()
        train_total += labels.size(0)

    # 검증 단계
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)

            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    # 결과 출력
    train_loss /= train_total
    val_loss /= val_total

    train_acc = train_correct / train_total
    val_acc = val_correct / val_total

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f}, "
        f"Train Acc: {train_acc:.4f}, "
        f"Val Loss: {val_loss:.4f}, "
        f"Val Acc: {val_acc:.4f}"
    )

In [16]:
from sklearn.metrics import classification_report, confusion_matrix

# Precision, Recall, F1-score와 혼동 행렬을 확인

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(
    all_labels,
    all_preds,
    target_names=["Normal", "Pneumonia"],
    digits=4
))

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

              precision    recall  f1-score   support

      Normal     0.9961    0.9478    0.9713       268
   Pneumonia     0.9823    0.9987    0.9904       776

    accuracy                         0.9856      1044
   macro avg     0.9892    0.9732    0.9809      1044
weighted avg     0.9858    0.9856    0.9855      1044

Confusion Matrix:
[[254  14]
 [  1 775]]


In [17]:
from torch.utils.data import DataLoader
from google.colab import files

# 테스트 데이터셋 생성
test_dataset = ChestXrayDataset(
    test_df,
    '/content/chest_xray/test',
    transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

# 테스트 데이터 예측
model.eval()
test_preds = []

with torch.no_grad():
    for images in test_loader:
        images = images.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        test_preds.extend(preds.cpu().numpy())

# 파일명과 예측값 연결
pred_df = pd.DataFrame({
    'file_name': test_df['file_name'].values,
    'label': test_preds
})

# 제출 양식에 맞춰 정렬
submission = sample_submission[['file_name']].copy()

submission['label'] = submission['file_name'].map(
    pred_df.set_index('file_name')['label']
)

# 결과 확인
print(submission.head())
print("\n제출 데이터 크기:", submission.shape)
print("\n예측 클래스별 개수:")
print(submission['label'].value_counts().sort_index())

# 누락 여부 확인
assert len(submission) == len(test_df)
assert submission['label'].notna().all()
assert set(submission['label'].unique()).issubset({0, 1})

# CSV 저장
output_path = '/content/submission_resnet18.csv'
submission.to_csv(output_path, index=False)

print("\nCSV 저장 완료:", output_path)

# 컴퓨터로 다운로드
files.download(output_path)

       file_name  label
0  test_0001.png      0
1  test_0002.png      0
2  test_0003.png      1
3  test_0004.png      1
4  test_0005.png      1

제출 데이터 크기: (624, 2)

예측 클래스별 개수:
label
0    251
1    373
Name: count, dtype: int64

CSV 저장 완료: /content/submission_resnet18.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>